# Lab 04: Pandas Lambdas, `apply`/`map`, and Reshaping (Wide ↔ Tall)

**Duration:** ~60 minutes  
**Dataset:** `SampleData.csv` (already uploaded to this environment)

> This notebook is self-contained. Run cells in order. If you restart, re-run all cells above your current position.

## 0) Setup

In [40]:
import pandas as pd
import numpy as np

# Load data (file is in the working directory)
df = pd.read_csv("SampleData.csv")

# Parse and enrich date features
df["Invoice_date"] = pd.to_datetime(df["Invoice_date"], errors="coerce")
df["Year"] = df["Invoice_date"].dt.year
df["Month"] = df["Invoice_date"].dt.month
df["YearMonth"] = df["Invoice_date"].dt.to_period("M").astype(str)

# Convert Account_no to string type
df["Account_no"] = df["Account_no"].astype('string')

df.head()

,Account_no,LocationID,CustomerID,ProductID,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill,Year,Month,YearMonth
0,1002001,L1001,P01,P01,302,2020-03-05,12.0,0.03,9.06,2020,3,2020-03
1,1002001,L1001,P01,P01,520,2020-04-06,12.0,0.03,15.60,2020,4,2020-04
2,1002001,L1001,P01,P01,619,2020-05-04,12.0,0.03,18.57,2020,5,2020-05
3,1002001,L1001,P01,P01,614,2020-06-03,12.0,0.03,18.42,2020,6,2020-06
4,1002001,L1001,P01,P01,1389,2020-07-03,12.0,0.03,41.67,2020,7,2020-07


## Part A — Lambda + `apply`/`map` in practice

### A1) Price tiers via `map`

In [41]:
price_to_tier = {0.025: "Low", 0.030: "Medium", 0.045: "High"}  # adjust if your file has different values
df["PriceTier"] = df["Price"].map(price_to_tier).fillna("Other")
df["PriceTier"].value_counts()

PriceTier
Medium    96
High      96
Low       96
Name: count, dtype: int64

### A2) Usage bucket via `apply` + `lambda`

In [42]:
def usage_bucket(kwh: float) -> str:
    if kwh < 200: 
        return "Low"
    elif kwh < 500:
        return "Medium"
    else:
        return "High"

df["UsageBucket"] = df["Billed_usage_kwh"].apply(lambda x: usage_bucket(x))
df["UsageBucket"].value_counts()

UsageBucket
High      200
Medium     88
Name: count, dtype: int64

### A3) Row-wise calculations with `DataFrame.apply(axis=1)` + `.assign`

In [43]:
df = df.assign(
    Variable_charge=lambda _df: _df["Billed_usage_kwh"] * _df["Price"],
    Recalc_Bill=lambda _df: _df["Base_charge"] + _df["Variable_charge"],
    Bill_Diff=lambda _df: (_df["Recalc_Bill"] - _df["Bill"]).round(2)
)

df["Recalc_Bill_rowwise"] = df.apply(
    lambda r: r["Base_charge"] + r["Billed_usage_kwh"] * r["Price"], axis=1
)

df[["Bill", "Recalc_Bill", "Bill_Diff"]].head()

,Bill,Recalc_Bill,Bill_Diff
0,9.06,21.06,12.0
1,15.60,27.60,12.0
2,18.57,30.57,12.0
3,18.42,30.42,12.0
4,41.67,53.67,12.0


### A4) Chaining transformations with `.assign()`

In [44]:
summer_months = {6, 7, 8, 9}
q90 = df["Bill"].quantile(0.90)

print(f"90th percentile of Bill: {q90:.2f}")

df = (
    df
    .assign(
        IsSummer=lambda _df: _df["Month"].isin(summer_months),
        IsHighBill=lambda _df: _df["Bill"] >= q90,
        SummerDiscount=lambda _df: np.where(_df["IsSummer"] & (_df["PriceTier"]=="High"), 0.05, 0.0),
        Discounted_Bill=lambda _df: (1 - _df["SummerDiscount"]) * _df["Recalc_Bill"]
    )
)

#print(df[["Month", "PriceTier", "IsSummer", "IsHighBill", "SummerDiscount", "Discounted_Bill"]].head())
df.loc[df["PriceTier"] == "High", ["Month", "PriceTier", "IsSummer", "SummerDiscount", "Discounted_Bill"]].head()

90th percentile of Bill: 57.39


,Month,PriceTier,IsSummer,SummerDiscount,Discounted_Bill
12,3,High,False,0.00,33.46500
13,4,High,False,0.00,47.55000
14,5,High,False,0.00,37.15500
15,6,High,True,0.05,21.53175
16,7,High,True,0.05,80.27025


### A5) Translating codes to labels with `map`

In [45]:
prod_map = {"P01": "Standard", "P02": "Peak", "P03": "OffPeak"}
df["ProductLabel"] = df["ProductID"].map(prod_map).fillna(df["ProductID"])
df["ProductLabel"].value_counts()

ProductLabel
Standard    96
Peak        96
OffPeak     96
Name: count, dtype: int64

## Part B — Reshaping: Wide ↔ Tall

### B1) Tall monthly summary

In [46]:
tall = (
    df
    .groupby(["Account_no", "YearMonth", "ProductID"], as_index=False)
    .agg(
        Monthly_kwh=("Billed_usage_kwh", "sum"),
        Monthly_bill=("Bill", "sum")
    )
    .sort_values(["Account_no", "YearMonth", "ProductID"])
)
tall.head()

,Account_no,YearMonth,ProductID,Monthly_kwh,Monthly_bill
0,1002001,2020-03,P01,302,9.06
1,1002001,2020-04,P01,520,15.60
2,1002001,2020-05,P01,619,18.57
3,1002001,2020-06,P01,614,18.42
4,1002001,2020-07,P01,2724,81.72


### B2) Wide usage with `pivot_table`

In [47]:
wide_kwh = (
    tall
    .pivot_table(
        index=["Account_no", "YearMonth"],
        columns="ProductID",
        values="Monthly_kwh",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)
wide_kwh.columns.name = None
wide_kwh.head()

,Account_no,YearMonth,P01,P02,P03
0,1002001,2020-03,302,0,0
1,1002001,2020-04,520,0,0
2,1002001,2020-05,619,0,0
3,1002001,2020-06,614,0,0
4,1002001,2020-07,2724,0,0


### B3) Back to tall with `melt` and validate round-trip

In [ ]:
# 1) Melt back to tall
tall_back = (
    wide_kwh
    .melt(id_vars=["Account_no", "YearMonth"], var_name="ProductID", value_name="Monthly_kwh")
    .sort_values(["Account_no", "YearMonth", "ProductID"])
    .reset_index(drop=True)
)

# 2) Align by keys and compare to original tall (treat missing combos as 0 in original)
keys = ["Account_no", "YearMonth", "ProductID"]

cmp = (
    tall_back
    .merge(
        tall[keys + ["Monthly_kwh"]],
        on=keys,
        how="left",
        suffixes=("_back", "")
    )
    .fillna({"Monthly_kwh": 0})
    .sort_values(keys, kind="stable")
    .reset_index(drop=True)
)

# display(cmp)

# 3) Validation result: True means round-trip preserved monthly usage
import numpy as np
np.allclose(cmp["Monthly_kwh_back"].to_numpy(), cmp["Monthly_kwh"].to_numpy())

,Account_no,YearMonth,ProductID,Monthly_kwh_back,Monthly_kwh
0,1002001,2020-03,P01,302,302.0
1,1002001,2020-03,P02,0,0.0
2,1002001,2020-03,P03,0,0.0
3,1002001,2020-04,P01,520,520.0
4,1002001,2020-04,P02,0,0.0
...,...,...,...,...,...
781,1002024,2020-12,P02,0,0.0
782,1002024,2020-12,P03,1349,1349.0
783,1002024,2021-01,P01,0,0.0
784,1002024,2021-01,P02,0,0.0


True

### B4) Wide with multiple values (MultiIndex columns)

In [50]:
wide_multi = (
    tall
    .pivot_table(
        index=["Account_no", "YearMonth"],
        columns="ProductID",
        values=["Monthly_kwh", "Monthly_bill"],
        aggfunc="sum",
        fill_value=0
    )
)
# Flatten columns
wide_multi.columns = [f"{val}_{pid}" for (val, pid) in wide_multi.columns]
wide_multi = wide_multi.reset_index()
wide_multi.head()

,Account_no,YearMonth,Monthly_bill_P01,Monthly_bill_P02,Monthly_bill_P03,Monthly_kwh_P01,Monthly_kwh_P02,Monthly_kwh_P03
0,1002001,2020-03,9.06,0.0,0.0,302,0,0
1,1002001,2020-04,15.60,0.0,0.0,520,0,0
2,1002001,2020-05,18.57,0.0,0.0,619,0,0
3,1002001,2020-06,18.42,0.0,0.0,614,0,0
4,1002001,2020-07,81.72,0.0,0.0,2724,0,0


### B5) `stack` / `unstack` alternative

In [51]:
tall_usage = tall.set_index(["Account_no", "YearMonth", "ProductID"])["Monthly_kwh"]

wide_via_unstack = tall_usage.unstack("ProductID", fill_value=0).reset_index()

tall_via_stack = (
    wide_via_unstack
    .set_index(["Account_no", "YearMonth"])
    .stack(future_stack=True)  # 👈 updated for pandas 2.1+
    .rename_axis(["Account_no", "YearMonth", "ProductID"])
    .rename("Monthly_kwh")
    .reset_index()
)

tall_via_stack.head()

,Account_no,YearMonth,ProductID,Monthly_kwh
0,1002001,2020-03,P01,302
1,1002001,2020-03,P02,0
2,1002001,2020-03,P03,0
3,1002001,2020-04,P01,520
4,1002001,2020-04,P02,0


## Part C — Mini-challenges / Stretch

**1) Percent contribution per product per month (tall).**

In [52]:
tall_pct = (
    tall
    .assign(Total_kwh=lambda _df: _df.groupby(["Account_no", "YearMonth"])["Monthly_kwh"].transform("sum"))
    .assign(Pct_of_month=lambda _df: (_df["Monthly_kwh"] / _df["Total_kwh"]).round(4))
)
tall_pct.head()

,Account_no,YearMonth,ProductID,Monthly_kwh,Monthly_bill,Total_kwh,Pct_of_month
0,1002001,2020-03,P01,302,9.06,302,1.0
1,1002001,2020-04,P01,520,15.60,520,1.0
2,1002001,2020-05,P01,619,18.57,619,1.0
3,1002001,2020-06,P01,614,18.42,614,1.0
4,1002001,2020-07,P01,2724,81.72,2724,1.0


**2) Detect missing product months.**

In [57]:
counts = tall.groupby(["Account_no", "YearMonth"])["ProductID"].nunique()
missing = counts[counts < 3].reset_index(name="n_products")
missing.head()

,Account_no,YearMonth,n_products
0,1002001,2020-03,1
1,1002001,2020-04,1
2,1002001,2020-05,1
3,1002001,2020-06,1
4,1002001,2020-07,1


**3) Friendly row label via row-wise `apply`.**

In [58]:
tall["Label"] = tall.apply(
    lambda r: f"A{r['Account_no']}-{r['YearMonth']}[{r['ProductID']}]", axis=1
)
tall[["Label", "Monthly_kwh"]].head()

,Label,Monthly_kwh
0,A1002001-2020-03[P01],302
1,A1002001-2020-04[P01],520
2,A1002001-2020-05[P01],619
3,A1002001-2020-06[P01],614
4,A1002001-2020-07[P01],2724


**4) Map codes to categories (practice).**

In [59]:
cust_map = {"P01": "Residential", "P02": "Commercial", "P03": "Industrial"}
df["CustomerType"] = df["CustomerID"].map(cust_map).fillna("Other")
df["CustomerType"].value_counts()

CustomerType
Residential    96
Commercial     96
Industrial     96
Name: count, dtype: int64

## (Optional) Save artifacts

In [60]:
artifacts = {
    "tall.csv": tall,
    "wide_kwh.csv": wide_kwh,
    "wide_multi.csv": wide_multi
}
for name, frame in artifacts.items():
    frame.to_csv(name, index=False)
list(artifacts.keys())

['tall.csv', 'wide_kwh.csv', 'wide_multi.csv']